In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [13]:
from clean.cleaning import clean_all
from feats.features import features_all
from utils import merge_train_test
import numpy as np
from utils import merge_train_test, split_train_test
# load data...

df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

df_all = merge_train_test(df_train, df_test)
print(df_all.columns)
# CLEANING

df_cleaned = clean_all(df_all)
print(df_cleaned.isna().sum())

# FE

df_final = features_all(df_cleaned)
print(df_final.columns)
print("=-" * 15)

# PRIPREMI ZA MODEL....
train_final, test_final = split_train_test(df_final)

Index(['ID', 'Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
       'Date_birth', 'Date_driving_licence', 'Distribution_channel',
       'Seniority', 'Policies_in_force', 'Max_policies', 'Max_products',
       'Lapse', 'Date_lapse', 'Payment', 'Premium', 'Cost_claims_year',
       'N_claims_year', 'N_claims_history', 'R_Claims_history', 'Type_risk',
       'Area', 'Second_driver', 'Year_matriculation', 'Power',
       'Cylinder_capacity', 'Value_vehicle', 'N_doors', 'Type_fuel', 'Length',
       'Weight', '_base'],
      dtype='object')
ID                          0
Date_start_contract         0
Date_last_renewal           0
Date_next_renewal           0
Date_birth                  0
Date_driving_licence        0
Distribution_channel        0
Seniority                   0
Policies_in_force           0
Max_policies                0
Max_products                0
Lapse                       0
Date_lapse              70408
Payment                     0
Premium             

In [14]:
train_final

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,doors_total_size,car_age,car_age_log,broj_prethodnih_semplova,max_prethodni,min_prethodni,mean_prethodni,ima_prethodni,cena_direktno_prethodne,mean_kroz_broj
0,1,05/11/2015,05/11/2015,2016-11-05,15/04/1956,20/03/1976,0,4,1,2,...,0.000,22,3.135494,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
1,1,05/11/2015,05/11/2016,2017-11-05,15/04/1956,20/03/1976,0,4,1,2,...,0.000,22,3.135494,1,5.409501,5.409501,5.409501,True,5.409501,5.409501
2,1,05/11/2015,05/11/2017,2018-11-05,15/04/1956,20/03/1976,0,4,2,2,...,0.000,22,3.135494,2,5.409501,5.369614,5.389558,True,5.369614,2.694779
4,2,26/09/2017,26/09/2017,2018-09-26,15/04/1956,20/03/1976,0,4,2,2,...,0.000,22,3.135494,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
6,3,29/11/2013,29/11/2015,2016-11-29,18/03/1975,10/07/1995,0,15,1,2,...,110472.375,13,2.639057,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105549,53497,10/11/2018,10/11/2018,2019-11-10,29/06/1961,28/12/1990,0,1,1,1,...,172352.800,9,2.302585,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
105550,53498,30/07/2018,30/07/2018,2019-07-30,25/07/1981,14/02/2007,0,1,1,1,...,175380.000,26,3.295837,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
105551,53499,16/08/2018,16/08/2018,2019-08-16,08/12/1976,29/11/2017,0,1,1,1,...,167400.000,13,2.639057,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
105552,53500,21/11/2018,21/11/2018,2019-11-21,01/04/1974,05/10/2011,0,1,1,1,...,72521.250,27,3.332205,0,0.000000,0.000000,0.000000,False,0.000000,0.000000


In [7]:
train_final["broj_polisa_po_ID"] = train_final.groupby("ID").transform("size")

In [11]:
train_final["ima_drugu_polisu"] = (train_final["broj_polisa_po_ID"] > 1).astype(int)

In [17]:
def split_by_unique_policies(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    # radimo copy da ne menjamo orginale
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    # pretpostavka: ID kolona je 'ID', zameni ako je drugačije
    id_col = "ID"
    
    # ------------- za trening -------------
    df_train["broj_polisa_po_ID"] = df_train.groupby(id_col).transform("size")
    df_train["ima_drugu_polisu"] = (df_train["broj_polisa_po_ID"] > 1).astype(int)
    
    train1 = df_train[df_train["ima_drugu_polisu"] == 1].copy()
    train0 = df_train[df_train["ima_drugu_polisu"] == 0].copy()
    
    # ------------- za test -------------
    df_test["broj_polisa_po_ID"] = df_test.groupby(id_col).transform("size")
    df_test["ima_drugu_polisu"] = (df_test["broj_polisa_po_ID"] > 1).astype(int)
    
    test1 = df_test[df_test["ima_drugu_polisu"] == 1].copy()
    test0 = df_test[df_test["ima_drugu_polisu"] == 0].copy()
    
    # ------------- obriši pomoćne kolone -------------
    cols_to_drop = ["broj_polisa_po_ID", "ima_drugu_polisu"]
    
    for df in [train1, train0, test1, test0]:
        df.drop(columns=cols_to_drop, errors="ignore", inplace=True)
    
    return train1, train0, test1, test0

In [18]:
train_nonunique, train_unique, test_nonunique, test_unique = split_by_unique_policies(train_final,test_final)

In [23]:
train_unique

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,doors_total_size,car_age,car_age_log,broj_prethodnih_semplova,max_prethodni,min_prethodni,mean_prethodni,ima_prethodni,cena_direktno_prethodne,mean_kroz_broj
4,2,26/09/2017,26/09/2017,2018-09-26,15/04/1956,20/03/1976,0,4,2,2,...,0.000,22,3.135494,0,0.0,0.0,0.0,False,0.0,0.0
13,5,12/05/2017,12/05/2017,2018-05-12,10/07/1973,05/07/1993,0,3,2,2,...,0.000,40,3.713572,0,0.0,0.0,0.0,False,0.0,0.0
22,9,24/10/2013,24/10/2016,2017-10-24,22/10/1949,17/05/1995,1,6,1,3,...,116128.125,13,2.639057,0,0.0,0.0,0.0,False,0.0,0.0
24,10,13/01/2017,13/01/2017,2018-01-13,22/10/1949,17/05/1995,1,6,3,3,...,175567.500,13,2.639057,0,0.0,0.0,0.0,False,0.0,0.0
26,11,10/05/2017,10/05/2017,2018-05-10,22/10/1949,17/05/1995,1,6,3,3,...,170304.375,13,2.639057,0,0.0,0.0,0.0,False,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105549,53497,10/11/2018,10/11/2018,2019-11-10,29/06/1961,28/12/1990,0,1,1,1,...,172352.800,9,2.302585,0,0.0,0.0,0.0,False,0.0,0.0
105550,53498,30/07/2018,30/07/2018,2019-07-30,25/07/1981,14/02/2007,0,1,1,1,...,175380.000,26,3.295837,0,0.0,0.0,0.0,False,0.0,0.0
105551,53499,16/08/2018,16/08/2018,2019-08-16,08/12/1976,29/11/2017,0,1,1,1,...,167400.000,13,2.639057,0,0.0,0.0,0.0,False,0.0,0.0
105552,53500,21/11/2018,21/11/2018,2019-11-21,01/04/1974,05/10/2011,0,1,1,1,...,72521.250,27,3.332205,0,0.0,0.0,0.0,False,0.0,0.0


In [24]:
train_nonunique

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,doors_total_size,car_age,car_age_log,broj_prethodnih_semplova,max_prethodni,min_prethodni,mean_prethodni,ima_prethodni,cena_direktno_prethodne,mean_kroz_broj
0,1,05/11/2015,05/11/2015,2016-11-05,15/04/1956,20/03/1976,0,4,1,2,...,0.000,22,3.135494,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
1,1,05/11/2015,05/11/2016,2017-11-05,15/04/1956,20/03/1976,0,4,1,2,...,0.000,22,3.135494,1,5.409501,5.409501,5.409501,True,5.409501,5.409501
2,1,05/11/2015,05/11/2017,2018-11-05,15/04/1956,20/03/1976,0,4,2,2,...,0.000,22,3.135494,2,5.409501,5.369614,5.389558,True,5.369614,2.694779
6,3,29/11/2013,29/11/2015,2016-11-29,18/03/1975,10/07/1995,0,15,1,2,...,110472.375,13,2.639057,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
7,3,29/11/2013,29/11/2016,2017-11-29,18/03/1975,10/07/1995,0,15,1,2,...,110472.375,13,2.639057,1,5.943324,5.943324,5.943324,True,5.943324,5.943324
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105429,53406,19/08/2016,19/08/2017,2018-08-19,06/04/1955,09/04/1974,0,3,1,1,...,106176.000,19,2.995732,1,5.690596,5.690596,5.690596,True,5.690596,5.690596
105454,53421,28/11/2016,28/11/2016,2017-11-28,01/01/1978,11/09/1997,0,3,1,1,...,103669.280,18,2.944439,0,0.000000,0.000000,0.000000,False,0.000000,0.000000
105455,53421,28/11/2016,28/11/2017,2018-11-28,01/01/1978,11/09/1997,0,3,1,1,...,103669.280,18,2.944439,1,5.927539,5.927539,5.927539,True,5.927539,5.927539
105469,53430,28/11/2016,28/11/2016,2017-11-28,08/08/1956,19/05/1975,0,3,1,3,...,0.000,11,2.484907,0,0.000000,0.000000,0.000000,False,0.000000,0.000000


In [27]:
df_all

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Second_driver,Year_matriculation,Power,Cylinder_capacity,Value_vehicle,N_doors,Type_fuel,Length,Weight,_base
0,1,05/11/2015,05/11/2015,05/11/2016,15/04/1956,20/03/1976,0,4,1,2,...,0,2004,80,599,7068.00,0,P,NaN,190,train
1,1,05/11/2015,05/11/2016,05/11/2017,15/04/1956,20/03/1976,0,4,1,2,...,0,2004,80,599,7068.00,0,P,NaN,190,train
2,1,05/11/2015,05/11/2017,05/11/2018,15/04/1956,20/03/1976,0,4,2,2,...,0,2004,80,599,7068.00,0,P,NaN,190,train
3,2,26/09/2017,26/09/2017,26/09/2018,15/04/1956,20/03/1976,0,4,2,2,...,0,2004,80,599,7068.00,0,P,NaN,190,train
4,3,29/11/2013,29/11/2015,29/11/2016,18/03/1975,10/07/1995,0,15,1,2,...,0,2013,85,1229,16030.00,5,P,3.999,1105,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53473,13/11/2018,13/11/2018,13/11/2019,22/05/1981,01/03/2014,1,1,1,1,...,0,2001,112,2184,18600.00,5,D,4.197,1298,test
105551,53474,05/10/2018,05/10/2018,05/10/2019,05/03/1985,26/04/2003,0,1,2,2,...,0,2002,170,2685,40823.11,4,D,4.818,1650,test
105552,53477,22/11/2018,22/11/2018,22/11/2019,10/09/1963,22/10/2014,1,1,1,1,...,0,2010,90,1560,19120.00,5,D,4.276,1228,test
105553,53496,11/09/2018,11/09/2018,11/09/2019,26/07/1984,28/04/2017,0,1,1,1,...,0,1998,11,505,10217.21,3,P,NaN,400,test
